# Experiment: Biohub Prefix-Aware Hybrid Candidate

**Question:** can we recover the strong classical `44b6` behavior while retaining the
learned model's `6bba` edge-recall gain?

Exact held-out evidence supports a deterministic prefix split:

- `44b6`: NMS-3.8 classical `0.942567` vs learned `0.883791`.
- `6bba`: learned `0.836847` vs NMS-3.8 classical `0.802713`.
- Combined hybrid exact score: `0.842616`, above both full-pipeline validations.

This notebook only composes two completed, reproducible kernel outputs. It does not
rerun inference and it does not submit automatically.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

KAGGLE_CLASSICAL = Path('/kaggle/input/biohub-nms-3-8-submission-candidate/submission.csv')
KAGGLE_LEARNED = Path('/kaggle/input/biohub-learned-unet-ilp-candidate/submission.csv')
LOCAL_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
LOCAL_CLASSICAL = LOCAL_ROOT / 'references/nms38-v1-output/submission.csv'
LOCAL_LEARNED = LOCAL_ROOT / 'references/learned-candidate-v1-output/submission.csv'
WORKING = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/tmp/biohub-prefix-hybrid')
WORKING.mkdir(parents=True, exist_ok=True)

CLASSICAL_PATH = KAGGLE_CLASSICAL if KAGGLE_CLASSICAL.exists() else LOCAL_CLASSICAL
LEARNED_PATH = KAGGLE_LEARNED if KAGGLE_LEARNED.exists() else LOCAL_LEARNED
assert CLASSICAL_PATH.exists(), f'Missing classical kernel output: {CLASSICAL_PATH}'
assert LEARNED_PATH.exists(), f'Missing learned kernel output: {LEARNED_PATH}'
print('Classical:', CLASSICAL_PATH)
print('Learned:', LEARNED_PATH)


## Plan

1. Load both completed submission files.
2. Select classical rows for datasets beginning `44b6_`.
3. Select learned rows for datasets beginning `6bba_`.
4. Reassign the required global row ID.
5. Validate schema, coverage, graph endpoints, time steps, and lineage degrees.


In [ ]:
EXPECTED_COLUMNS = [
    'id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x',
    'source_id', 'target_id',
]
classical = pd.read_csv(CLASSICAL_PATH)
learned = pd.read_csv(LEARNED_PATH)
assert classical.columns.tolist() == EXPECTED_COLUMNS
assert learned.columns.tolist() == EXPECTED_COLUMNS
assert set(classical['dataset']) == set(learned['dataset'])

classical_rows = classical[classical['dataset'].str.startswith('44b6_')].copy()
learned_rows = learned[learned['dataset'].str.startswith('6bba_')].copy()
hybrid = pd.concat([classical_rows, learned_rows], ignore_index=True)
hybrid['id'] = np.arange(len(hybrid), dtype=np.int64)
print('Datasets:', sorted(hybrid['dataset'].unique()))
print('Rows:', f'{len(hybrid):,}')


## Structural validation

The competition metric is unavailable for hidden test labels, but all submission and
tracking-graph invariants can be checked exactly before writing the candidate.


In [ ]:
assert hybrid.columns.tolist() == EXPECTED_COLUMNS
assert hybrid['id'].is_unique
assert not hybrid.isna().any().any()
assert set(hybrid['row_type']) == {'node', 'edge'}
assert set(hybrid['dataset']) == set(classical['dataset'])

stats = []
for dataset, group in hybrid.groupby('dataset', sort=True):
    nodes = group[group['row_type'] == 'node']
    edges = group[group['row_type'] == 'edge']
    node_ids = set(nodes['node_id'].astype(np.int64))
    sources = edges['source_id'].astype(np.int64)
    targets = edges['target_id'].astype(np.int64)
    assert nodes['node_id'].is_unique
    assert set(sources).issubset(node_ids)
    assert set(targets).issubset(node_ids)
    node_t = nodes.set_index(nodes['node_id'].astype(np.int64))['t'].astype(np.int64)
    assert ((targets.map(node_t) - sources.map(node_t)) == 1).all()
    out_degree = sources.value_counts()
    in_degree = targets.value_counts()
    assert int(out_degree.max()) <= 2
    assert int(in_degree.max()) <= 1
    stats.append({
        'dataset': dataset,
        'source': 'classical_nms38' if dataset.startswith('44b6_') else 'learned_unet_ilp',
        'nodes': int(len(nodes)),
        'edges': int(len(edges)),
        'divisions': int((out_degree == 2).sum()),
    })

stats_df = pd.DataFrame(stats)
display(stats_df)


In [ ]:
OUTPUT = WORKING / 'submission.csv'
hybrid.to_csv(OUTPUT, index=False)
stats_df.to_csv(WORKING / 'hybrid_test_stats.csv', index=False)
manifest = {
    'strategy': 'classical_44b6_learned_6bba',
    'validation_score': 0.8426160044335661,
    'rows': int(len(hybrid)),
    'nodes': int((hybrid['row_type'] == 'node').sum()),
    'edges': int((hybrid['row_type'] == 'edge').sum()),
    'divisions': int(stats_df['divisions'].sum()),
    'datasets': sorted(hybrid['dataset'].unique()),
    'schema_validated': True,
}
(WORKING / 'hybrid_candidate_manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print('Wrote:', OUTPUT)


## Decision

Submit only after the output manifest and per-dataset statistics match the expected
source split. The public leaderboard score is the next evidence gate.
